# Phase 5: Training Loop Mastery

## What you'll learn
- The complete training loop pattern
- Loss functions: CrossEntropy, MSE, BCE
- Optimizers: SGD, Adam, AdamW
- Learning rate schedulers
- Validation loop and metrics
- Early stopping
- Gradient clipping

---

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
import torchvision
import torchvision.transforms as transforms

## 5.1 Setup: MNIST Data + Model

We'll use MNIST throughout this notebook to demonstrate the full training pipeline.

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# Data
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

full_train = torchvision.datasets.MNIST('./data', train=True, download=True, transform=transform)
test_set = torchvision.datasets.MNIST('./data', train=False, download=True, transform=transform)

# Split train into train + val
train_set, val_set = random_split(full_train, [50000, 10000])

train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
val_loader = DataLoader(val_set, batch_size=256, shuffle=False)
test_loader = DataLoader(test_set, batch_size=256, shuffle=False)

print(f"Train: {len(train_set)}, Val: {len(val_set)}, Test: {len(test_set)}")

In [ ]:
class MNISTNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.net = nn.Sequential(
            nn.Linear(784, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        return self.net(self.flatten(x))

## 5.2 Loss Functions

The loss function measures how wrong the model's predictions are.

| Loss | Use Case | Input |
|------|----------|-------|
| `CrossEntropyLoss` | Multi-class classification | Raw logits (no softmax!) |
| `MSELoss` | Regression | Continuous values |
| `BCEWithLogitsLoss` | Binary classification | Raw logits (no sigmoid!) |

In [ ]:
# CrossEntropyLoss — combines LogSoftmax + NLLLoss
criterion = nn.CrossEntropyLoss()

logits = torch.tensor([[2.0, 1.0, 0.1]])  # raw model output
target = torch.tensor([0])                  # correct class = 0
loss = criterion(logits, target)
print(f"CrossEntropy loss: {loss.item():.4f}")

# MSELoss — for regression
mse = nn.MSELoss()
pred = torch.tensor([2.5, 0.0, 2.1])
true = torch.tensor([3.0, -0.5, 2.0])
print(f"MSE loss: {mse(pred, true).item():.4f}")

# BCEWithLogitsLoss — binary classification
bce = nn.BCEWithLogitsLoss()
logit = torch.tensor([0.5, -1.0, 2.0])
target_bin = torch.tensor([1.0, 0.0, 1.0])
print(f"BCE loss: {bce(logit, target_bin).item():.4f}")

## 5.3 Optimizers

Optimizers update model parameters based on gradients.

| Optimizer | Notes |
|-----------|-------|
| `SGD` | Simple, needs momentum for good results |
| `Adam` | Adaptive LR, good default |
| `AdamW` | Adam + proper weight decay (recommended) |

In [ ]:
model = MNISTNet().to(device)

# SGD with momentum
optimizer_sgd = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

# Adam (most popular default)
optimizer_adam = optim.Adam(model.parameters(), lr=0.001)

# AdamW (recommended for modern training)
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)

print(f"Using: {optimizer.__class__.__name__}")

## 5.4 The Complete Training Loop

This is the **core pattern** you'll use in every PyTorch project:

```
for each epoch:
    for each batch:
        1. Forward pass  → compute predictions
        2. Compute loss  → how wrong are we?
        3. Backward pass → compute gradients
        4. Update weights → optimizer.step()
        5. Zero gradients → optimizer.zero_grad()
```

In [ ]:
model = MNISTNet().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3)

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()  # enable dropout, batchnorm training mode
    total_loss = 0
    correct = 0
    total = 0

    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()          # 5. zero gradients
        outputs = model(inputs)        # 1. forward pass
        loss = criterion(outputs, labels)  # 2. compute loss
        loss.backward()                # 3. backward pass
        optimizer.step()               # 4. update weights

        total_loss += loss.item() * inputs.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += inputs.size(0)

    return total_loss / total, correct / total

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()  # disable dropout, batchnorm eval mode
    total_loss = 0
    correct = 0
    total = 0

    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        total_loss += loss.item() * inputs.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += inputs.size(0)

    return total_loss / total, correct / total

# Train!
for epoch in range(5):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)
    print(f"Epoch {epoch+1}: train_loss={train_loss:.4f} train_acc={train_acc:.4f} | val_loss={val_loss:.4f} val_acc={val_acc:.4f}")

## 5.5 Learning Rate Schedulers

Schedulers adjust the learning rate during training for better convergence.

In [ ]:
model = MNISTNet().to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-3)

# StepLR — decay by gamma every step_size epochs
scheduler_step = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)

# CosineAnnealing — smooth decay following cosine curve
scheduler_cosine = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)

# OneCycleLR — warmup then decay (great for fast training)
scheduler_onecycle = optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=0.01, steps_per_epoch=len(train_loader), epochs=10
)

# Usage in training loop:
# for epoch in range(epochs):
#     train_one_epoch(...)
#     scheduler.step()  # call after each epoch (or step for OneCycleLR)

print(f"Current LR: {optimizer.param_groups[0]['lr']}")

## 5.6 Early Stopping

Stop training when validation loss stops improving to prevent overfitting.

In [ ]:
class EarlyStopping:
    def __init__(self, patience=5, min_delta=0.0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = float('inf')

    def __call__(self, val_loss):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
            return False  # don't stop
        self.counter += 1
        return self.counter >= self.patience  # stop if patience exceeded

# Usage:
early_stop = EarlyStopping(patience=3)
model = MNISTNet().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3)

for epoch in range(50):
    train_loss, _ = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)
    print(f"Epoch {epoch+1}: val_loss={val_loss:.4f} val_acc={val_acc:.4f}")

    if early_stop(val_loss):
        print(f"Early stopping at epoch {epoch+1}")
        break

## 5.7 Gradient Clipping

Prevents exploding gradients (common in RNNs). Clips gradient norms to a max value.

In [ ]:
model = MNISTNet().to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()
max_norm = 1.0

# Single training step with gradient clipping
inputs, labels = next(iter(train_loader))
inputs, labels = inputs.to(device), labels.to(device)

optimizer.zero_grad()
outputs = model(inputs)
loss = criterion(outputs, labels)
loss.backward()

# Clip gradients AFTER backward, BEFORE step
grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm)
print(f"Gradient norm before clipping: {grad_norm:.4f}")

optimizer.step()

## ✅ Phase 5 Summary

You now have the complete training toolkit:
- Full training loop: forward → loss → backward → step → zero_grad
- Loss functions for classification and regression
- Optimizers (AdamW is a great default)
- LR schedulers for better convergence
- Early stopping to prevent overfitting
- Gradient clipping for stability

**Next → Phase 6: Save/Load & Inference**